## Online Retail Transactions

**Understanding the Dataset**

The online retail csv file is an extensive collection of data relating to ecommerce transactions. This dataset provides a detailed view of sales activities within the online retail sector, covering numerous essential attributes necessary for a quantitative understanding of consumer behavior and the overall business performance.

**The Columns**:
* **InvoiceNo**	A unique identification number assigned to each transaction. (Numeric)
* **StockCode**	A unique identification code assigned to each product sold by the retailer. (Numeric)
* **Description**	A brief description of the product sold. (Text)
* **Quantity**	The number of units of the product sold in each transaction. (Numeric)
* **InvoiceDate**	The exact date and time when the transaction occurred. (Date/Time)
* **UnitPrice**	The price per unit of the product sold. (Numeric)
* **Country**	The country where the customer resides. (Text)

Source: https://www.kaggle.com/datasets/thedevastator/online-retail-transaction-records

**Import Pyspark and start a SparkSession**

In [1]:
from pyspark.sql import SparkSession

In [2]:
# start a spark session
spark = (SparkSession.builder
         .master('local')
         .appName("online_retail_project")
         .getOrCreate())

# create a spark context
sc = spark.sparkContext

**Step 1: Load the csv file** 

In [13]:
# load data
data = sc.textFile('dataset/retail_dataset.csv')

In [8]:
print(type(data))

<class 'pyspark.rdd.RDD'>


confirmed that the data is of type rdd, that means our data was read successfully.

In [25]:
# load header only
header = data.first()
print("Header: \n", header, "\n")

# load every line that is not equal to the header
rows = data.filter(lambda line: line != header)
top_10 = rows.take(10)

#view first 10 rows
for row in top_10:
    print(row)

Header: 
 index,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country 

0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
5,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,12/1/2010 8:26,7.65,17850.0,United Kingdom
6,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,12/1/2010 8:26,4.25,17850.0,United Kingdom
7,536366,22633,HAND WARMER UNION JACK,6,12/1/2010 8:28,1.85,17850.0,United Kingdom
8,536366,22632,HAND WARMER RED POLKA DOT,6,12/1/2010 8:28,1.85,17850.0,United Kingdom
9,536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,12/1/2010 8:34,1.69,13047.0,United Kingdom


**Step 2: Cleaning Data**

Now let's get a sense of our dataset and possible data quality issues

In [28]:
# check total rows of data
total_rows = rows.count()
print(f"There are a total of {total_rows} rows of data. Excluding headers")

There are a total of 541909 rows of data. Excluding headers


In [29]:
## count number of rows per column (nulls, missing, unusual)

# first, let's split each row by the delimiter
split_data = rows.map(lambda row: row.split(','))

# check the number of rows per column - if there is a row with more columns then we have a problem
col_counts = split_data.map(lambda row: len(row)).countByValue()
print(col_counts)

defaultdict(<class 'int'>, {9: 537113, 10: 3729, 11: 1067})


In [32]:
print("Number of columns are: ", len(header.split(",")))

Number of columns are:  9


From the column counts:
* 537k rows have 9 columns which is the original number of columns
* 3729 rows have 10 columns which means there are extra commas
* 1067 rows have 11 columns, just like the above. 

With the extra commas, there is proof that the data is malformed. We will inspect the malformed rows

In [34]:
## identify missing values per column

num_cols = len(header.split(','))

for i in range(num_cols):
    missing_count = split_data.filter(lambda row: row[i].strip() == "").count()
    print(f'Column {i} missing values: {missing_count}')

Column 0 missing values: 0
Column 1 missing values: 0
Column 2 missing values: 0
Column 3 missing values: 1454
Column 4 missing values: 0
Column 5 missing values: 0
Column 6 missing values: 0
Column 7 missing values: 133391
Column 8 missing values: 1365


columns 3 (Quantity), 7 (UnitPrice), 8 (Country) all have missing values. 